In [ ]:
!pip install anthropic==0.86.0

In [ ]:
import anthropic
import time
import csv
from tqdm import tqdm
import argparse
from openai import OpenAI

## **CONFIGURATION**

In [ ]:
OPENAI_API_KEY    = "OPENAI_API_KEY"
ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"

MODELS = {
    "gpt-5.4":           "openai",
    "claude-opus-4-6":   "anthropic"
}

# Reflection settings
MAX_REFLECT  = 2     # fixed number of reflection rounds
DELTA        = 0.10  # minimum score change to continue reflecting

openai_client    = OpenAI(
    api_key = OPENAI_API_KEY
)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
def call_llm(prompt, model):

    provider = MODELS.get(model)

    if provider is None:
      print(f"  Unknown model: {model}")
      return None

    if provider == "openai":
      response = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
      if response.choices[0].finish_reason == "length":
          print("WARNING: GPT output truncated!")
      return response.choices[0].message.content

    elif provider == "anthropic":
      response = anthropic_client.messages.create(
            model=model,
            temperature=0,
            max_tokens=16000,
            messages=[{"role": "user", "content": prompt}]
      )
      if response.stop_reason == "max_tokens":
          print("WARNING: Claude output truncated!")
      return response.content[0].text


    return None

##AUTO-COT STEP GENERATION

In [ ]:
import os

REL_RULE_REF = """Matching:
- A candidate relationship MATCHES a gold relationship if it connects the
  same pair of classes (exact or semantic class-name match), regardless of
  arrow direction.

Correctness rule:
- Matched AND relationship type (association, aggregation, composition,
  inheritance, realization, dependency) equals the gold type
  AND multiplicities are consistent with the gold -> CORRECT
- Not matched, but supported by the problem description with an
  appropriate type and plausible multiplicities -> CORRECT
- Otherwise -> INCORRECT

Multiplicities:
- Only association, aggregation and composition carry multiplicities;
  for inheritance, realization and dependency ignore them.
- Judge multiplicities by meaning, not notation: equivalent notations
  are the same (e.g. * = 0..* , 1 = 1..1).
- Multiplicities are inconsistent only if they change the meaning,
  e.g. one vs. many at the same end.
- Direction is not considered.
- Real-world knowledge is NOT a valid source."""

REL_RULE_NOREF = """Correctness rule:
- A candidate relationship is CORRECT if the problem description supports
  a relationship between these two classes, its type (association,
  aggregation, composition, inheritance, realization, dependency) is
  appropriate, AND its multiplicities are consistent with the problem
  description.
- Otherwise -> INCORRECT

Multiplicities:
- Only association, aggregation and composition carry multiplicities;
  for inheritance, realization and dependency ignore them.
- Judge multiplicities by meaning, not notation: equivalent notations
  are the same (e.g. * = 0..* , 1 = 1..1).
- If the problem description does not specify a multiplicity, accept any
  plausible one; mark it incorrect only if it contradicts the description
  (e.g. one vs. many).
- Direction is not considered.
- The ONLY valid reference is the problem description.
- Real-world knowledge is NOT a valid source."""


def generate_cot_steps_no_gold(model):
    """Generate CoT steps for RELATIONSHIP evaluation WITHOUT gold reference."""
    prompt = f"""You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram given
only a problem description.

The evaluator will apply this rule:
{REL_RULE_NOREF}

Generate steps that cover:
- How to identify expected relationships between classes
  from the problem description
- How to match candidate relationships to expected ones
  by class pair (regardless of direction)
- How to measure Relationship Completeness (0-1):
  fraction of expected relationships present in the candidate
- How to measure Relationship Correctness (0-1):
  correct candidate relationships / total candidate relationships,
  applying the rule above
- How to handle extra relationships (not supported by the problem)
- How to handle missing relationships
- How to handle relationships involving unmatched classes

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (no gold) using {model}...")
    return call_llm(prompt, model)


def generate_cot_steps_with_gold(model):
    """Generate CoT steps for RELATIONSHIP evaluation WITH gold reference."""
    prompt = f"""You are an expert in UML class diagram evaluation.

Generate detailed step-by-step instructions to evaluate
the RELATIONSHIPS in a PlantUML candidate diagram against
a gold reference diagram and a problem description.

The evaluator will apply this rule:
{REL_RULE_REF}

Generate steps that cover:
- How to enumerate all relationships in the gold diagram
  (two class endpoints, type, multiplicities)
- How to enumerate all relationships in the candidate diagram
  in the same format
- How to match candidate relationships to gold relationships
  by class pair (regardless of direction)
- How to handle naming variants in class names
  (e.g. OrderMgr vs OrderManager)
- How to measure Relationship Completeness (0-1):
  matched gold relationships / total gold relationships
- How to measure Relationship Correctness (0-1):
  correct candidate relationships / total candidate relationships,
  applying the rule above
- How to handle unmatched candidate relationships
  (check the problem description before marking incorrect)
- How to handle missing relationships (in gold but not in candidate)
- How to handle relationships involving unmatched classes

Evaluation Steps:"""

    print(f"\n  Generating CoT steps (with gold) using {model}...")
    return call_llm(prompt, model)


def get_cot_steps(model, with_gold):
    """Generate the steps ONCE per model and reuse them (S2/S3 and S5/S6 share steps)."""
    tag  = "with_gold" if with_gold else "no_gold"
    path = f"cot_steps_relationships_{tag}_{model}.txt"
    if os.path.exists(path):
        print(f"  Reusing steps from {path}")
        with open(path, encoding="utf-8") as f:
            return f.read()
    steps = generate_cot_steps_with_gold(model) if with_gold else generate_cot_steps_no_gold(model)
    with open(path, "w", encoding="utf-8") as f:
        f.write(steps)
    print(f"  Saved to {path}")
    return steps

## Reflection

In [ ]:
def needs_reflection(round_num, current_scores,previous_scores=None):
    """
    Decide whether to run a reflection round.

    Rules:
    - Round 1: always runs (previous_scores is None,
      no delta to compare yet)
    - Round 2: only runs if max score delta between
      initial scores and round 1 scores > DELTA
    - Never exceeds MAX_REFLECT rounds

    Args:
        round_num:       current round number (1-based)
        current_scores:  scores from most recent round
        previous_scores: scores from the round before
                         (None for round 1)
    """
    # Never exceed max reflection rounds
    if round_num > MAX_REFLECT:
        return False

    # Round 1 always runs — no previous scores to compare
    if previous_scores is None:
        return True

    # Round 2+ — only run if scores are still changing
    deltas = [
        abs(current_scores[k] - previous_scores[k])
        for k in current_scores
        if current_scores.get(k) is not None
        and previous_scores.get(k) is not None
    ]

    if not deltas:
        return False

    return max(deltas) > DELTA

## PROMPT

In [ ]:
def s1_prompt(problem, candidate):
    """S1 — Simple baseline. No CoT, no gold."""
    return f"""You are evaluating the RELATIONSHIPS in a
PlantUML candidate diagram.

Definitions:
- Completeness (0-1): fraction of expected relationships
  present in the candidate diagram.
- Correctness (0-1): fraction of candidate relationships
  that are correct.

{REL_RULE_NOREF}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s2_prompt(problem, candidate, cot_steps):
    """S2 and S3 base scoring prompt. No gold reference."""
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s3_reflection_prompt(problem, candidate, cot_steps, previous_scores, round_num):
    """S3 — Reflection prompt. No gold reference."""
    comp = previous_scores['rel_completeness']
    corr = previous_scores['rel_correctness']

    return f"""You previously evaluated the RELATIONSHIPS in
this diagram (reflection round {round_num}) and produced:

- Relationship Completeness: {comp}
- Relationship Correctness:  {corr}

Please reflect carefully using ONLY the problem
description as your reference:

For Relationship Completeness:
1. Re-read the problem description carefully.
2. List every relationship you believe should exist
   between classes, based on the problem description.
3. For each expected relationship, check if a
   corresponding one exists in the candidate diagram.
4. Recompute: completeness = found / expected
5. Revise your score if needed.

For Relationship Correctness:
1. List every relationship in the candidate diagram.
2. For each one, apply this rule:
{REL_RULE_NOREF}
3. Count: correct / total candidate relationships
4. Revise your score if needed.

{cot_steps}

Problem Description:
{problem}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s4_prompt(problem, gold, candidate):
    """S4 — Reference only. No CoT."""
    return f"""You are evaluating the RELATIONSHIPS in a
PlantUML candidate diagram against a gold reference.

Definitions:
- Completeness (0-1): fraction of gold relationships
  present in the candidate.
  Formula: matched / total gold relationships
- Correctness (0-1): fraction of candidate relationships
  that are correct.
  Formula: correct / total candidate relationships

{REL_RULE_REF}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s5_prompt(problem, gold, candidate, cot_steps):
    """S5 and S6 base scoring prompt. With gold reference."""
    return f"""You are an expert evaluator of UML class diagrams.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""


def s6_reflection_prompt(problem, gold, candidate, cot_steps, previous_scores, round_num):
    """S6 — Reflection prompt. With gold reference."""
    comp = previous_scores['rel_completeness']
    corr = previous_scores['rel_correctness']

    return f"""You previously evaluated the RELATIONSHIPS in
this diagram (reflection round {round_num}) and produced:

- Relationship Completeness: {comp}
- Relationship Correctness:  {corr}

Please reflect carefully using the gold reference.

For Relationship Completeness:
1. List every relationship in the gold diagram with class pair.
2. For each gold relationship, check if a matching
   one exists in the candidate (match by class pair).
3. Count: matched / total gold relationships
4. Verify and revise your completeness score.

For Relationship Correctness:
1. List every relationship in the candidate diagram.
2. For each one, apply this rule:
{REL_RULE_REF}
3. Count: correct / total candidate relationships
4. Verify and revise your correctness score.

{cot_steps}

Problem Description:
{problem}

Gold Reference Diagram:
{gold}

Candidate Diagram:
{candidate}

Refined Evaluation Form (scores ONLY):
- Relationship Completeness:
- Relationship Correctness:"""

##SCORE PARSER

In [ ]:
import re

def parse_rel_scores(response):
    """
    Extract relationship completeness and correctness
    from LLM response text.
    Clamps values to [0, 1].
    Returns None for any score that cannot be parsed.
    """
    scores = {
        'rel_completeness': None,
        'rel_correctness':  None
    }

    if response is None:
        return scores

    # Remove markdown bold markers before processing
    clean_response = response.replace("**", "")

    for line in clean_response.strip().split("\n"):
        line_lower = line.lower()

        # Find a number in the line (int or float, optional %)
        match = re.search(r'[\=\:]\s*([\d\.]+)\s*%?', line)
        if not match:
            continue

        try:
            value = float(match.group(1))

            # Handle percentage format (e.g. 83.3% → 0.833)
            if "%" in line:
                value = value / 100

            # Clamp to [0, 1]
            value = max(0.0, min(1.0, value))

            if "relationship completeness" in line_lower:
                scores['rel_completeness'] = value
            elif "relationship correctness" in line_lower:
                scores['rel_correctness'] = value

        except ValueError:
            continue

    return scores

##EVALUATOR

In [ ]:
def evaluate_relationships(scenario, problem, candidate, model,
                           gold=None, cot_steps_no_gold=None,
                           cot_steps_with_gold=None):
    """
    Evaluate RELATIONSHIP dimension for one candidate
    under a given scenario.

    Returns dict:
        rel_completeness    : float or None
        rel_correctness     : float or None
        reflected           : bool
        reflection_rounds   : int
    """
    reflected         = False
    reflection_rounds = 0

    # ── S1 ──────────────────────────────────────────────
    if scenario == 's1':
        prompt   = s1_prompt(problem, candidate)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S2 ──────────────────────────────────────────────
    elif scenario == 's2':
        prompt   = s2_prompt(
                     problem, candidate, cot_steps_no_gold)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S3 — Auto-CoT + Reflection ──────────────────────
    # Uses same CoT steps as S2 (no gold)
    elif scenario == 's3':
        # Initial scoring — same as S2
        prompt          = s2_prompt(
                            problem, candidate, cot_steps_no_gold)
        response        = call_llm(prompt, model)
        scores          = parse_rel_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s3_reflection_prompt(
                problem, candidate, cot_steps_no_gold,
                previous_scores, round_num
            )
            response = call_llm(ref_prompt, model)
            scores   = parse_rel_scores(response)

    # ── S4 ──────────────────────────────────────────────
    elif scenario == 's4':
        prompt   = s4_prompt(problem, gold, candidate)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S5 ──────────────────────────────────────────────
    elif scenario == 's5':
        prompt   = s5_prompt(
                     problem, gold, candidate,
                     cot_steps_with_gold)
        response = call_llm(prompt, model)
        scores   = parse_rel_scores(response)

    # ── S6 — Reference + CoT + Reflection ───────────────
    elif scenario == 's6':
        # Initial scoring — same as S5
        prompt          = s5_prompt(
                            problem, gold, candidate,
                            cot_steps_with_gold)
        response        = call_llm(prompt, model)
        scores          = parse_rel_scores(response)
        previous_scores = None

        for round_num in range(1, MAX_REFLECT + 1):
            if not needs_reflection(round_num, scores,
                                    previous_scores):
                break

            reflected         = True
            reflection_rounds += 1
            previous_scores   = scores.copy()

            ref_prompt = s6_reflection_prompt(
                problem, gold, candidate, cot_steps_with_gold,
                previous_scores, round_num
            )
            response = call_llm(ref_prompt, model)
            scores   = parse_rel_scores(response)

    else:
        print(f"  Unknown scenario: {scenario}")
        scores = {
            'rel_completeness': None,
            'rel_correctness':  None
        }

    scores['reflected']         = reflected
    scores['reflection_rounds'] = reflection_rounds
    return scores


##DATA LOADING

In [ ]:
import chardet

def load_gold(filepath):
    with open(filepath, 'rb') as f:
        detected = chardet.detect(f.read())
    encoding = detected['encoding'] or 'windows-1252'

    gold = {}
    with open(filepath, newline='', encoding=encoding) as f:
        reader = csv.DictReader(f)
        for row in reader:
            gold[row['id']] = {
                'problem':  row['problem'],
                'solution': row['solution']
            }
    return gold


def load_candidates(filepath):
    """
    Load candidates CSV.
    Columns: id, student_answer,
             score1 (class_completeness),
             score2 (class_correctness),
             score3 (attr_completeness),
             score4 (attr_correctness),
             score5 (rel_completeness),
             score6 (rel_correctness)

    id directly matches gold file id.
    """
    candidates = []
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            candidates.append({
                'id':      row['id'],
                'diagram': row['student_answer'],
                'human_scores': {
                    'class_completeness': float(row['score1']),
                    'class_correctness':  float(row['score2']),
                    'attr_completeness':  float(row['score3']),
                    'attr_correctness':   float(row['score4']),
                    'rel_completeness':   float(row['score5']),
                    'rel_correctness':    float(row['score6'])
                }
            })
    return candidates



In [ ]:
def save_results(results, output_filepath):
    """Save LLM scores + human scores to CSV."""
    fieldnames = [
        'candidate_id', 'scenario', 'model',
        'reflected', 'reflection_rounds',
        'human_rel_completeness', 'human_rel_correctness',
        'llm_rel_completeness',   'llm_rel_correctness'
    ]
    with open(output_filepath, 'w', newline='',
              encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\n  Results saved to: {output_filepath}")

In [ ]:
def run_relationship_evaluation(gold_file, candidate_file,
                                scenario, model, output_file):
    """
    Main loop — evaluates RELATIONSHIP dimension for all
    candidates under a given scenario and model.
    """
    print(f"\n{'='*55}")
    print(f"  CD-EVAL — Relationship Dimension")
    print(f"  Scenario        : {scenario.upper()}")
    print(f"  Model           : {model}")
    print(f"  Max reflections : {MAX_REFLECT}")
    print(f"  Delta threshold : {DELTA}")
    print(f"{'='*55}")

    # Load data
    print("\nLoading data...")
    gold       = load_gold(gold_file)
    candidates = load_candidates(candidate_file)
    print(f"  Gold scenarios : {len(gold)}")
    print(f"  Candidates     : {len(candidates)}")

    # Generate CoT steps once if needed
    # S2 and S3 share the same no-gold steps
    # S5 and S6 share the same with-gold steps
    cot_steps_no_gold   = None
    cot_steps_with_gold = None

    if scenario in ['s2', 's3']:
        cot_steps_no_gold = get_cot_steps(model, with_gold=False)

    elif scenario in ['s5', 's6']:
        cot_steps_with_gold = get_cot_steps(model, with_gold=True)

    # Evaluation loop
    results = []
    print(f"\nEvaluating {len(candidates)} candidates...\n")

    for candidate in tqdm(candidates):

        cid          = candidate['id']
        diagram      = candidate['diagram']
        human_scores = candidate['human_scores']

        # id directly matches gold
        if cid not in gold:
            print(f"\n  Warning: no gold found for "
                  f"id={cid} — skipping.")
            continue

        problem  = gold[cid]['problem']
        solution = gold[cid]['solution']

        # Run evaluation
        llm_scores = evaluate_relationships(
            scenario            = scenario,
            problem             = problem,
            candidate           = diagram,
            model               = model,
            gold                = solution,
            cot_steps_no_gold   = cot_steps_no_gold,
            cot_steps_with_gold = cot_steps_with_gold
        )

        # Collect result row
        results.append({
            'candidate_id':    cid,
            'scenario':        scenario,
            'model':           model,
            'reflected':       llm_scores['reflected'],
            'reflection_rounds': llm_scores['reflection_rounds'],
            # human scores (relationship dimension only)
            'human_rel_completeness':
                human_scores['rel_completeness'],
            'human_rel_correctness':
                human_scores['rel_correctness'],
            # llm scores
            'llm_rel_completeness':
                llm_scores['rel_completeness'],
            'llm_rel_correctness':
                llm_scores['rel_correctness']
        })

        time.sleep(0.5)  # avoid rate limits

    save_results(results, output_file)

    # Summary
    reflected_count = sum(1 for r in results if r['reflected'])
    print(f"\n  Total evaluated     : {len(results)}")
    print(f"  Reflection triggered: {reflected_count} "
          f"({100*reflected_count//max(len(results),1)}%)")

In [ ]:
gold_file = "/content/drive/MyDrive/Paper5LLMasEval/GoldDataset.csv"
candidate_file= "/content/drive/MyDrive/Paper5LLMasEval/CandidatesDataset.csv"
scenario= "s6"
model="claude-opus-4-6"
output_file="/content/drive/MyDrive/Paper5LLMasEval/ClaudeRelS6.csv"

run_relationship_evaluation(gold_file, candidate_file, scenario, model, output_file)


  CD-EVAL — Relationship Dimension
  Scenario        : S6
  Model           : claude-opus-4-6
  Max reflections : 2
  Delta threshold : 0.1

Loading data...
  Gold scenarios : 13
  Candidates     : 13
  Reusing steps from cot_steps_relationships_with_gold_claude-opus-4-6.txt

Evaluating 13 candidates...



100%|██████████| 13/13 [16:14<00:00, 74.96s/it]



  Results saved to: /content/drive/MyDrive/Paper5LLMasEval/ClaudeRelS6.csv

  Total evaluated     : 13
  Reflection triggered: 13 (100%)


In [ ]:
if __name__ == "__main__":

    parser = argparse.ArgumentParser(
        description="CD-EVAL: Relationship dimension evaluator"
    )
    parser.add_argument(
        "--gold", required=True,
        help="Path to gold CSV (columns: id, problem, solution)"
    )
    parser.add_argument(
        "--candidates", required=True,
        help="Path to candidates CSV"
    )
    parser.add_argument(
        "--scenario", required=True,
        choices=['s1', 's2', 's3', 's4', 's5', 's6'],
        help="Evaluation scenario"
    )
    parser.add_argument(
        "--model", default="gpt-4",
        choices=list(MODELS.keys()),
        help="LLM to use as evaluator"
    )
    parser.add_argument(
        "--output", required=True,
        help="Path to output CSV file"
    )

    args = parser.parse_args()

    run_relationship_evaluation(
        gold_file      = args.gold,
        candidate_file = args.candidates,
        scenario       = args.scenario,
        model          = args.model,
        output_file    = args.output
    )
